# Making list of filenames

In [1]:
import matplotlib.pyplot as plt
from obspy import read_events
from obspy import UTCDateTime
import numpy as np
import pandas as pd
import scipy.stats as stats
import wget
import csv

In [2]:
# Catalog Data
big10_catalog = read_events("Big10_Greece_Seismicity.xml")
filenames = []

j=1
for event in big10_catalog: # For each earthquake

    # Read earthquake data
    origin = event.preferred_origin() or event.origins[0]
    event_time = origin.time
    focal_mech = event.preferred_focal_mechanism() or event.focal_mechanisms[0]
    moment_tensor = focal_mech.moment_tensor
    nodal_planes = focal_mech.nodal_planes
    plane1 = nodal_planes.nodal_plane_1
    plane2 = nodal_planes.nodal_plane_2
    
    mag = event.preferred_magnitude() or event.magnitudes[0]
    mag = float(mag.mag)

    for plane in (plane1,plane2):
        filenames.append(f"Greece_EQ{j}_M{mag}_{plane}_{event_time}.csv")
    j=j+1

    df = pd.DataFrame(filenames)
    df.to_csv("filenames.csv", index=False, header=None)

# Downloading the files

In [3]:
# Catalog Data
filenames = pd.read_csv("filenames.csv", names=['col'], header=None)
filenames=filenames['col'].tolist()
print(filenames)

['Greece_EQ1_M6.72_NodalPlane(strike=201.0, dip=44.0, rake=55.0)_2006-01-08T11:35:00.300000Z.csv', 'Greece_EQ1_M6.72_NodalPlane(strike=66.0, dip=55.0, rake=119.0)_2006-01-08T11:35:00.300000Z.csv', 'Greece_EQ2_M6.85_NodalPlane(strike=332.0, dip=6.0, rake=120.0)_2008-02-14T10:09:29.000000Z.csv', 'Greece_EQ2_M6.85_NodalPlane(strike=121.0, dip=85.0, rake=87.0)_2008-02-14T10:09:29.000000Z.csv', 'Greece_EQ3_M6.54_NodalPlane(strike=337.0, dip=5.0, rake=127.0)_2008-02-14T12:09:02.700000Z.csv', 'Greece_EQ3_M6.54_NodalPlane(strike=120.0, dip=86.0, rake=87.0)_2008-02-14T12:09:02.700000Z.csv', 'Greece_EQ4_M6.76_NodalPlane(strike=339.0, dip=3.0, rake=130.0)_2013-10-12T13:11:56.400000Z.csv', 'Greece_EQ4_M6.76_NodalPlane(strike=119.0, dip=88.0, rake=88.0)_2013-10-12T13:11:56.400000Z.csv', 'Greece_EQ5_M6.86_NodalPlane(strike=73.0, dip=85.0, rake=-177.0)_2014-05-24T09:25:18.800000Z.csv', 'Greece_EQ5_M6.86_NodalPlane(strike=343.0, dip=87.0, rake=-5.0)_2014-05-24T09:25:18.800000Z.csv', 'Greece_EQ6_M6.5_N

In [4]:
EQ=1
for i in range(20):
    filename=filenames[i]
    print(EQ)
    print(f"GNSSVerify/EQ{EQ}.{2-(i+1)%2}/")
    print(filename)
    relevant_stations = pd.read_csv(f"Finite/{filename}")
    sta_ids = relevant_stations["Station_ID"]
    for station in sta_ids:
        url = f"https://geodesy.unr.edu/gps_timeseries/IGS20/tenv3/EU/{station.upper()}.EU.tenv3"
        wget.download(url, out=f"GNSSVerify/EQ{EQ}.{2-(i+1)%2}/")
    if ((i+1)%2==0):
        EQ=EQ+1

1
GNSSVerify/EQ1.1/
Greece_EQ1_M6.72_NodalPlane(strike=201.0, dip=44.0, rake=55.0)_2006-01-08T11:35:00.300000Z.csv
100% [............................................................................] 300696 / 3006961
GNSSVerify/EQ1.2/
Greece_EQ1_M6.72_NodalPlane(strike=66.0, dip=55.0, rake=119.0)_2006-01-08T11:35:00.300000Z.csv
100% [............................................................................] 300696 / 3006962
GNSSVerify/EQ2.1/
Greece_EQ2_M6.85_NodalPlane(strike=332.0, dip=6.0, rake=120.0)_2008-02-14T10:09:29.000000Z.csv
100% [............................................................................] 608736 / 6087362
GNSSVerify/EQ2.2/
Greece_EQ2_M6.85_NodalPlane(strike=121.0, dip=85.0, rake=87.0)_2008-02-14T10:09:29.000000Z.csv
100% [............................................................................] 608736 / 6087363
GNSSVerify/EQ3.1/
Greece_EQ3_M6.54_NodalPlane(strike=337.0, dip=5.0, rake=127.0)_2008-02-14T12:09:02.700000Z.csv
100% [.......................

# Computing Coseismic Deformation from GNSS time-series

In [21]:
%%bash

# Bayesian Code

t1=$2 # Start Time
t2=$3 # End Time

dirEXE=./GNSSVerify # Contains Executables

dirIN=/GNSSVerify/EQ1.1 # Input Folder
dirOUT=/GNSSVerify/EQ1.1 # Output Folder

cd $dirEXE # Go to executables folder
shopt -s nullglob # Recognize Wildcards
files=("$dirIN"/*.EU.tenv3)

# For every .tenv3 file in the folder
for f in "${files[@]}"; do
    sta=$(basename "$f" .tenv3) # Station Code/ Identifier
    rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run

    # Reads file "$f", filters in range [t1,t2], and extracts 1:Time and 2:North to serv.inp
    gawk -v tin=$t1 -v tout=$t2 '{if($3 >= tin && $3 <= tout) print $3,$11}' "$f" > serv.inp
    octave -q < input_cycleslip.m # Find the Cycle Slip
    cp serv.bayes $dirOUT/$sta.N.bayes.out # Save in output directory
    cp serv.p_tau $dirOUT/$sta.N.bayes.p_tau # Save in output directory
    
    rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run

    # Reads file "$f", filters in range [t1,t2], and extracts 1:Time and 3:East to serv.inp
    gawk -v tin=$t1 -v tout=$t2 '{if($3 >= tin && $3 <= tout) print $3,$9}' "$f" > serv.inp
    octave -q < input_cycleslip.m # Find the Cycle Slip
    cp serv.bayes $dirOUT/$sta.E.bayes.out # Save in output directory
    cp serv.p_tau $dirOUT/$sta.E.bayes.p_tau # Save in output directory
    
    rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run

    # Reads file "$f", filters in range [t1,t2], and extracts 1:Time and 4:Up to serv.inp
    gawk -v tin=$t1 -v tout=$t2 '{if($3 >= tin && $3 <= tout) print $3,$13}' "$f" > serv.inp
    octave -q < input_cycleslip.m # Find the Cycle Slip
    cp serv.bayes $dirOUT/$sta.U.bayes.out # Save in output directory
    cp serv.p_tau $dirOUT/$sta.U.bayes.p_tau # Save in output directory

    echo "$f"
done